# 06 — Population Initialization

**Maps to:** `report/Chapters/Task2.tex` §`T2:Init`.  
**Ticket:** TICKET-06.

Random vs. nearest-neighbour-seeded vs. hybrid.

Below blocks of code, these are two functions used, such as `random_population(pop_size, n_cities, rng)` and `nearest_neighbour_seed(dist_matrix, start_city)`.

For `random_population(pop_size, n_cities, rng)`, this function generates a population of `pop_size` chromosomes where each chromosome is a randomly shuffled permutation of city indices from 0 to n_cities-1. `nearest_neighbour_seed(dist_matrix, start_city)` constructs a single tour using the nearest-neighbour heuristic. Starting from `start_city`, it repeatedly visits the closest unvisited city until all cities have been visited, forming a closed tour.

In [1]:
import numpy as np

from pathlib import Path
import tsplib95

In [2]:
def random_population(pop_size, n_cities, rng):
    return np.array([rng.permutation(n_cities) for _ in range(pop_size)])
    #raise NotImplementedError

def nearest_neighbour_seed(dist_matrix, start_city):
    n_cities  = len(dist_matrix)
    unvisited = set(range(n_cities))
    tour      = [start_city]
    unvisited.remove(start_city)

    while unvisited:
        current = tour[-1]
        nearest = min(unvisited, key=lambda city: dist_matrix[current, city])
        tour.append(nearest)
        unvisited.remove(nearest)

    return np.array(tour)
    #raise NotImplementedError

Below code is for hybrid (random population + nearest-neighbour-seeded) implementation:

In [3]:
def hybrid_population(pop_size, n_cities, dist_matrix, rng):
    half       = pop_size // 2
    random_pop = random_population(half, n_cities, rng)

    nn_tours = []
    for start_city in range(pop_size - half):
        tour = nearest_neighbour_seed(dist_matrix, start_city)
        nn_tours.append(tour)

    nn_pop = np.array(nn_tours)
    return np.vstack([random_pop, nn_pop])

## Unit Test

In this section, both functions will be verified aka checked before applying them to kroA100 through unit test.

In [4]:
rng = np.random.default_rng(42)

# unit test for random_population
pop = random_population(5, 4, rng)
print("Random population:")
for i, tour in enumerate(pop):
    print(f"  Tour {i}: {tour}")

try:
    assert pop.shape == (5, 4), f"Expected (5,4), got {pop.shape}"
    for tour in pop:
        assert len(set(tour)) == 4, f"Duplicate cities found: {tour}"
    print("random_population test passed.")
except AssertionError as e:
    print(f"random_population test failed: {e}")

# unit test for nearest_neighbour_seed
square_coords = np.array([[0,0],[1,0],[1,1],[0,1]], dtype=np.float64)
diff          = square_coords[:, np.newaxis, :] - square_coords[np.newaxis, :, :]
square_dist   = np.sqrt((diff**2).sum(axis=-1))

nn_tour = nearest_neighbour_seed(square_dist, start_city=0)
print(f"\nNearest-neighbour tour: {nn_tour}")

try:
    assert len(nn_tour) == 4, f"Expected 4 cities, got {len(nn_tour)}"
    assert len(set(nn_tour)) == 4, f"Duplicate cities found: {nn_tour}"
    print("nearest_neighbour_seed test passed.")
except AssertionError as e:
    print(f"nearest_neighbour_seed test failed: {e}")

Random population:
  Tour 0: [3 2 1 0]
  Tour 1: [3 2 0 1]
  Tour 2: [0 3 2 1]
  Tour 3: [2 3 0 1]
  Tour 4: [2 3 0 1]
random_population test passed.

Nearest-neighbour tour: [0 1 2 3]
nearest_neighbour_seed test passed.


For `Random Population`, five random tours were generated for four cities. Each tour contains cities 0, 1, 2, and 3 in a different random order, confirming that every city is visited exactly once. The order differs across tours because each one is generated independently using a random permutation.

For `Nearest-Neighbour Tour`, starting from city 0 at coordinate (0, 0), the algorithm always moves to the closest unvisited city. City 1 at (1, 0) is the nearest, followed by city 2 at (1, 1), then city 3 at (0, 1), before returning to city 0. This produces the tour [0, 1, 2, 3], which traces the perimeter of the square in order, which is the shortest possible path for this arrangement.

In here, both functions produced valid permutations with no duplicate or missing cities, confirming that the implementation is correct.

Below unit test is for hybrid:

In [5]:
rng = np.random.default_rng(42)

hybrid_pop = hybrid_population(6, 4, square_dist, rng)
print("Hybrid population:")
for i, tour in enumerate(hybrid_pop):
    print(f"  Tour {i}: {tour}")

try:
    assert hybrid_pop.shape == (6, 4), f"Expected (6,4), got {hybrid_pop.shape}"
    for tour in hybrid_pop:
        assert len(set(tour)) == 4, f"Duplicate cities found: {tour}"
    print("hybrid_population test passed.")
except AssertionError as e:
    print(f"hybrid_population test failed: {e}")

Hybrid population:
  Tour 0: [3 2 1 0]
  Tour 1: [3 2 0 1]
  Tour 2: [0 3 2 1]
  Tour 3: [0 1 2 3]
  Tour 4: [1 0 3 2]
  Tour 5: [2 1 0 3]
hybrid_population test passed.


## Demo on kroA100

In [6]:
def load_tsp(path):
    problem     = tsplib95.load(str(path))
    nodes       = list(problem.get_nodes())
    coords      = np.array(
        [problem.node_coords[node] for node in nodes],
        dtype=np.float64
    )
    diff        = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff**2).sum(axis=-1))
    return coords, dist_matrix

def tour_length(chromosome, dist_matrix):
    n     = len(chromosome)
    total = 0.0
    for i in range(n):
        city_a  = chromosome[i]
        city_b  = chromosome[(i + 1) % n]
        total  += dist_matrix[city_a, city_b]
    return total

coords, dist_matrix = load_tsp(Path('../data/TSP-dataset/kroA100.tsp'))

rng        = np.random.default_rng(42)
random_pop = random_population(10, len(coords), rng)
nn_tour    = nearest_neighbour_seed(dist_matrix, start_city=0)

random_lengths = [tour_length(t, dist_matrix) for t in random_pop]

print(f"Random population tour lengths:")
for i, length in enumerate(random_lengths):
    print(f"  Tour {i:2d}: {length:.2f}")

print(f"\nNearest-neighbour tour length : {tour_length(nn_tour, dist_matrix):.2f}")
print(f"Known optimal tour length     : 21,282")

Random population tour lengths:
  Tour  0: 166578.15
  Tour  1: 166765.28
  Tour  2: 174384.37
  Tour  3: 157074.39
  Tour  4: 177877.78
  Tour  5: 181990.18
  Tour  6: 169727.68
  Tour  7: 159527.50
  Tour  8: 169824.78
  Tour  9: 164494.06

Nearest-neighbour tour length : 26856.39
Known optimal tour length     : 21,282


Below demo is for hybrid function:

In [7]:
rng        = np.random.default_rng(42)
hybrid_pop = hybrid_population(10, len(coords), dist_matrix, rng) # will generate ten tour lengths

hybrid_lengths = [tour_length(t, dist_matrix) for t in hybrid_pop]

print("Hybrid population tour lengths:")
for i, length in enumerate(hybrid_lengths):
    half = len(hybrid_pop) // 2
    kind = "random" if i < half else "nearest-neighbour"
    print(f"  Tour {i:2d} ({kind:20s}): {length:.2f}")

Hybrid population tour lengths:
  Tour  0 (random              ): 166578.15
  Tour  1 (random              ): 166765.28
  Tour  2 (random              ): 174384.37
  Tour  3 (random              ): 157074.39
  Tour  4 (random              ): 177877.78
  Tour  5 (nearest-neighbour   ): 26856.39
  Tour  6 (nearest-neighbour   ): 26133.45
  Tour  7 (nearest-neighbour   ): 27033.40
  Tour  8 (nearest-neighbour   ): 26480.88
  Tour  9 (nearest-neighbour   ): 28152.70


The hybrid population combines both methods, where the first half consists of randomly generated tours and the second half consists of nearest-neighbour tours starting from different cities.

The results clearly show the difference in tour quality between the two methods. Random tours produce significantly longer distances ranging from approximately 157,000 to 178,000, as they visit cities in a completely random order with no consideration for proximity. 
In contrast, nearest-neighbour tours produce much shorter distances ranging from approximately 26,000 to 29,000, as each tour greedily selects the closest unvisited city at every step.

Although nearest-neighbour tours are considerably shorter than random tours, they are still notably longer than the known optimal tour length (kroA100) of 21,282. This gap demonstrates why the Genetic Algorithm is necessary, as it will further optimise these initial solutions through selection, crossover, and mutation to find a tour closer to the optimal.